# DSCI 552 - Homework 3
**Name:** Chengfeng Tang  
**GitHub Username:** tcf2021  
**USC ID:** 5488902262

## 1(a) - AReM data
I used the supplied AReM archive from the UCI Machine Learning Repository. It contains **88 instances** across **seven activities**. Each instance contains six sensor time series; the time column is an index and is not used as a predictor.

| Series | Original column |
|---|---|
| 1 | avg_rss12 |
| 2 | var_rss12 |
| 3 | avg_rss13 |
| 4 | var_rss13 |
| 5 | avg_rss23 |
| 6 | var_rss23 |

I preserved the original CSV files and cleaned them during loading. `bending2/dataset4.csv` uses spaces instead of commas. The last lines of `cycling/dataset9.csv` and `cycling/dataset14.csv` contain an extra trailing comma. I handled these delimiter issues without changing sensor values.

`sitting/dataset8.csv` has 479 observations and is missing the timestamp 13,500 ms. I inserted that timestamp and linearly interpolated each channel between 13,250 and 13,750 ms. This gives 480 observations per instance. The inserted row is an estimate, and the original file remains unchanged.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# Paths work from the notebook directory or the repository root.
ROOT = Path("..") if Path("../data/AReM").is_dir() else Path(".")
DATA_DIR = ROOT / "data" / "AReM"
OUTPUT_DIR = ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ACTIVITIES = ["bending1", "bending2", "cycling", "lying",
              "sitting", "standing", "walking"]
CHANNELS = ["avg_rss12", "var_rss12", "avg_rss13",
            "var_rss13", "avg_rss23", "var_rss23"]
STATISTICS = ["min", "max", "mean", "median", "std", "q1", "q3"]
EXPECTED_TIMES = np.arange(0, 120000, 250)

def load_instance(path):
    rows = []
    space_rows = 0
    trailing_commas = 0
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        space_rows += int("," not in line)
        trailing_commas += int(line.endswith(","))
        fields = re.split(r"[,\s]+", line.rstrip(", "))
        if len(fields) != 7:
            raise ValueError(f"Unexpected row in {path}: {line}")
        rows.append([float(value) for value in fields])
    frame = pd.DataFrame(rows, columns=["time"] + CHANNELS)
    assert np.isfinite(frame.to_numpy()).all()
    assert frame["time"].is_unique
    assert frame["time"].is_monotonic_increasing
    assert set(frame["time"]).issubset(set(EXPECTED_TIMES))
    missing = sorted(set(EXPECTED_TIMES) - set(frame["time"]))
    if missing:
        assert path.parent.name == "sitting"
        assert path.name == "dataset8.csv" and missing == [13500]
    clean = frame.set_index("time").reindex(EXPECTED_TIMES)
    clean = clean.interpolate(method="index", limit_area="inside")
    assert clean.shape == (480, 6)
    assert np.isfinite(clean.to_numpy()).all()
    audit = {"raw_rows": len(frame), "clean_rows": len(clean),
             "space_rows": space_rows,
             "trailing_commas": trailing_commas,
             "imputed_rows": len(missing)}
    return clean, audit

In [2]:
instances = {}
records = []
for activity in ACTIVITIES:
    paths = sorted((DATA_DIR / activity).glob("dataset*.csv"),
                   key=lambda p: int(p.stem.replace("dataset", "")))
    for path in paths:
        instance_id = len(records) + 1
        series, audit = load_instance(path)
        instances[instance_id] = series
        records.append({"instance": instance_id, "activity": activity,
                        "file": path.name, **audit})
manifest = pd.DataFrame(records).set_index("instance")
assert len(instances) == 88
print("Instances:", len(instances))
print("Channels per instance:", len(CHANNELS))
print("Rows after cleaning:", sorted(manifest.clean_rows.unique().tolist()))
print("\nFiles requiring cleaning:")
changed = manifest.query(
    "space_rows > 0 or trailing_commas > 0 or imputed_rows > 0"
)
print(changed[["activity", "file", "raw_rows", "imputed_rows"]]
      .to_string())

Instances: 88
Channels per instance: 6
Rows after cleaning: [480]

Files requiring cleaning:
          activity           file  raw_rows  imputed_rows
instance                                                 
11        bending2   dataset4.csv       480             0
22         cycling   dataset9.csv       480             0
27         cycling  dataset14.csv       480             0
51         sitting   dataset8.csv       479             1


## 1(b) - Training and test sets
I assigned datasets 1 and 2 from each bending folder to the test set, along with datasets 1, 2, and 3 from every other activity. All remaining instances form the training set. The split contains **69 training instances** and **19 test instances**. I used the numeric filename suffix to identify datasets and did not shuffle the files.

In [3]:
def assign_split(row):
    number = int(Path(row["file"]).stem.replace("dataset", ""))
    cutoff = 2 if row["activity"] in ["bending1", "bending2"] else 3
    return "test" if number <= cutoff else "train"

manifest["split"] = manifest.apply(assign_split, axis=1)
split_counts = pd.crosstab(manifest.activity, manifest.split)
split_counts = split_counts.reindex(ACTIVITIES)[["train", "test"]]
split_counts["total"] = split_counts.sum(axis=1)
split_counts.loc["Total"] = split_counts.sum()
assert split_counts.loc["Total", "train"] == 69
assert split_counts.loc["Total", "test"] == 19
print(split_counts.to_string())
manifest.to_csv(OUTPUT_DIR / "instance_manifest.csv")

split     train  test  total
activity                    
bending1      5     2      7
bending2      4     2      6
cycling      12     3     15
lying        12     3     15
sitting      12     3     15
standing     12     3     15
walking      12     3     15
Total        69    19     88


## 1(c)(i) - Common time-domain features
Common time-domain features summarize a signal's level, variation, distribution, and changes over time. Examples include:

| Property | Features |
|---|---|
| Level and extremes | Mean, median, minimum, maximum |
| Variation | Standard deviation, variance, range, interquartile range |
| Distribution | Quartiles, other quantiles, skewness, kurtosis |
| Magnitude | Root mean square, sum of squared values (energy) |
| Temporal behavior | Autocorrelation, trend slope, mean absolute successive change, peak counts, threshold-crossing counts |

These features are documented in the tsfresh feature-calculator reference [2]. The seven summary statistics extracted below describe the distribution of values, but do not preserve their temporal order. Features such as autocorrelation and successive changes capture additional information about that order.

## 1(c)(ii) - Extracted features
For each instance, I computed seven statistics for each of the six channels, giving **42 numeric features** and an **$88\times42$ feature matrix**. I used the original measurement scales.

I used sample standard deviation with `ddof=1`:

$$s=\sqrt{\frac{1}{m-1}\sum_{t=1}^{m}(x_t-\bar{x})^2},\qquad m=480.$$

Q1 and Q3 use linear interpolation at the 25th and 75th percentiles. Column names combine the statistic and channel number: for example, `mean_1` is the mean of the first channel and `q3_6` is the third quartile of the sixth channel.

The complete dataset is saved as `data/processed/features_all.csv`, with instance ID, activity, filename, and split as metadata. The training and test feature tables are also saved separately. The tables below show selected rows and all 42 feature values for the first instance.

In [4]:
def extract_features(frame):
    summaries = {
        "min": frame.min(), "max": frame.max(),
        "mean": frame.mean(), "median": frame.median(),
        "std": frame.std(ddof=1),
        "q1": frame.quantile(0.25, interpolation="linear"),
        "q3": frame.quantile(0.75, interpolation="linear")
    }
    return {f"{stat}_{i}": summaries[stat][channel]
            for i, channel in enumerate(CHANNELS, start=1)
            for stat in STATISTICS}

features = pd.DataFrame.from_dict(
    {i: extract_features(frame) for i, frame in instances.items()},
    orient="index"
)
features.index.name = "instance"
assert features.shape == (88, 42)
assert features.index.equals(manifest.index)
assert np.isfinite(features.to_numpy()).all()

metadata = manifest[["activity", "file", "split"]]
feature_table = metadata.join(features)
feature_table.to_csv(OUTPUT_DIR / "features_all.csv")
for split in ["train", "test"]:
    feature_table.loc[feature_table.split == split].to_csv(
        OUTPUT_DIR / f"features_{split}.csv")

print("Numeric feature matrix:", features.shape)
preview = features.loc[[1, 2, 3, 86, 87, 88],
                       ["min_1", "max_1", "mean_1", "median_1",
                        "q1_6", "q3_6"]]
print(preview.round(3).to_string())

Numeric feature matrix: (88, 42)
          min_1  max_1  mean_1  median_1  q1_6   q3_6
instance                                             
1         37.25  45.00  40.625     40.50  0.00  1.300
2         38.00  45.67  42.813     42.50  0.00  1.300
3         35.00  47.40  43.954     44.33  0.00  0.940
86        19.50  46.00  34.546     35.25  2.16  4.335
87        23.50  46.25  34.873     35.25  2.17  4.500
88        19.25  44.00  34.473     35.00  2.12  4.375


In [5]:
first_instance = pd.DataFrame({
    stat: [features.loc[1, f"{stat}_{i}"] for i in range(1, 7)]
    for stat in STATISTICS
}, index=range(1, 7))
first_instance.index.name = "Series"
print("Instance 1: bending1/dataset1.csv")
print(first_instance.round(3).to_string())

Instance 1: bending1/dataset1.csv
          min    max    mean  median    std     q1     q3
Series                                                   
1       37.25  45.00  40.625   40.50  1.477  39.25  42.00
2        0.00   1.30   0.359    0.43  0.323   0.00   0.50
3        4.00  29.50  19.041   19.25  4.463  16.00  23.25
4        0.00   7.23   0.833    0.50  0.966   0.00   1.12
5       27.25  38.25  34.311   35.00  2.188  33.00  36.00
6        0.00   1.92   0.571    0.43  0.583   0.00   1.30


## 1(c)(iii) - Standard deviations and 90% bootstrap intervals
I estimated the sample standard deviation of each feature **across the 88 instances**. This is different from the within-instance `std` feature: for example, the standard deviation of `mean_1` measures how much channel 1's instance means vary across recordings.

I used a percentile bootstrap with **20,000 resamples** and random seed **42**. Each resample draws 88 complete feature rows with replacement. I calculated all 42 sample standard deviations for each resample and used the 5th and 95th percentiles as the endpoints of each 90% confidence interval. Resampling entire rows preserves the relationships between features; I did not resample the 480 time points within a recording.

These are individual intervals for the pooled activity mixture, treating recordings as independent sampling units. They are not simultaneous confidence intervals for all 42 features. I used all 88 instances for this descriptive calculation; the train/test assignments remain available for subsequent modeling.

In [6]:
values = features.to_numpy()
n, p = values.shape
B = 20000
BATCH = 500
rng = np.random.default_rng(42)
bootstrap_sds = np.empty((B, p))
for start in range(0, B, BATCH):
    size = min(BATCH, B - start)
    indices = rng.integers(0, n, size=(size, n))
    bootstrap_sds[start:start + size] = values[indices].std(
        axis=1, ddof=1)

lower, upper = np.quantile(bootstrap_sds, [0.05, 0.95], axis=0)
ci_table = pd.DataFrame({
    "SD": values.std(axis=0, ddof=1),
    "90% lower": lower, "90% upper": upper
}, index=features.columns)
ci_table.index.name = "Feature"
assert np.isfinite(ci_table.to_numpy()).all()
assert (ci_table["90% lower"] <= ci_table["90% upper"]).all()
ci_table.to_csv(OUTPUT_DIR / "feature_sd_bootstrap_90.csv")
print("Resamples:", B, "Confidence level: 90%", "Seed: 42")

Resamples: 20000 Confidence level: 90% Seed: 42


In [7]:
print(ci_table.round(4).to_string())

              SD  90% lower  90% upper
Feature                               
min_1     9.5700     8.2564    10.7609
max_1     4.3944     3.3359     5.2958
mean_1    5.3357     4.7090     5.8808
median_1  5.4401     4.7929     6.0081
std_1     1.7722     1.5710     1.9512
q1_1      6.1536     5.5755     6.6503
q3_1      5.1389     4.3371     5.8364
min_2     0.0000     0.0000     0.0000
max_2     5.0627     4.6283     5.4015
mean_2    1.5742     1.3972     1.7056
median_2  1.4122     1.2385     1.5436
std_2     0.8841     0.8024     0.9417
q1_2      0.9464     0.8300     1.0351
q3_2      2.1253     1.8937     2.2953
min_3     2.9565     2.7554     3.1080
max_3     4.8751     4.1675     5.4656
mean_3    4.0084     3.4101     4.4990
median_3  4.0364     3.4242     4.5447
std_3     0.9467     0.7628     1.1248
q1_3      4.2207     3.6293     4.7069
q3_3      4.1716     3.5430     4.6964
min_4     0.0000     0.0000     0.0000
max_4     2.1836     1.9712     2.3617
mean_4    1.1661     1.07

`min_2` and `min_4` are zero in all 88 recordings, so their estimated standard deviations and bootstrap intervals are exactly zero. Every resample is constant for these two features; this does not prove that their population variation must be zero.

## 1(c)(iv) - Three selected feature types
I selected **mean, standard deviation, and maximum**, giving 18 features across the six channels.

- **Mean** captures the overall signal level and can reflect differences in posture or sensor configuration during an activity.
- **Standard deviation** captures within-recording variability and can help distinguish relatively steady activities from activities with more movement.
- **Maximum** captures high-amplitude observations that may be averaged out by the mean.

These three types describe complementary aspects of the signal. I chose them by their interpretation rather than by comparing classification accuracy on the test set. Maximum can be sensitive to unusual readings, so its usefulness would need to be checked during later model validation.

In [8]:
selected_types = ["mean", "std", "max"]
selected_columns = [f"{stat}_{i}" for i in range(1, 7)
                    for stat in selected_types]
selected = features[selected_columns]
metadata.join(selected).to_csv(OUTPUT_DIR / "features_selected.csv")
print("Selected feature matrix:", selected.shape)
for i in range(1, 7):
    print(", ".join(f"{stat}_{i}" for stat in selected_types))

Selected feature matrix: (88, 18)
mean_1, std_1, max_1
mean_2, std_2, max_2
mean_3, std_3, max_3
mean_4, std_4, max_4
mean_5, std_5, max_5
mean_6, std_6, max_6


## 2 - ISLR 3.7.4
### (a) - Training RSS when the true relationship is linear
The cubic model has training RSS less than or equal to that of the linear model. The linear model is a special case of the cubic model with $\beta_2=\beta_3=0$. Least squares minimizes RSS over a larger set of possible fits for the cubic model, so adding those terms cannot increase its minimum training RSS. The cubic model will usually fit some noise and obtain a smaller value.

### (b) - Test RSS when the true relationship is linear
I would generally expect the linear model to have lower test RSS. It already represents the true relationship, while the extra quadratic and cubic coefficients introduce estimation variance without reducing model bias. This is an expectation over repeated samples, not a guarantee for every particular test set.

### (c) - Training RSS when the true relationship is nonlinear
The cubic model still has training RSS less than or equal to that of the linear model. The nesting argument from part (a) holds regardless of the true relationship between $X$ and $Y$.

### (d) - Test RSS when the true relationship is nonlinear
There is not enough information to determine which model has lower test RSS. If the curvature is strong and well approximated by a cubic, the reduction in bias can outweigh the increase in variance. If the relationship is close to linear, or the extra terms do not approximate it well, the linear model may predict better. The comparison depends on the form and strength of the nonlinearity, the noise level, and the resulting bias-variance trade-off.

## References
1. UCI Machine Learning Repository, [Activity Recognition system based on Multisensor data fusion (AReM)](https://archive.ics.uci.edu/dataset/366/activity+recognition+system+based+on+multisensor+data+fusion+arem). Data supplied with the assignment.
2. tsfresh documentation, [Overview on extracted features](https://tsfresh.readthedocs.io/en/stable/text/list_of_features.html).
3. SciPy documentation, [Bootstrap confidence intervals](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html). I implemented the percentile procedure directly with NumPy.
4. James, Witten, Hastie, and Tibshirani, *An Introduction to Statistical Learning with Applications in R*, first edition, Section 3.7, Exercise 4. [Book website](https://www.statlearning.com/).